# EDFA Digital Twin — Results Visualisation

Reads per-seed outputs and reproduces the training / test diagnostics used in the paper.

All cells are idempotent (re-runnable, no disk mutation). Populate `results/seed_<N>/` first with e.g. `python scripts/run_seeds.py --seeds 42 43 44 --aggregate-after`, then execute this notebook top-to-bottom.

**Layout (post multi-seed refactor)**:
- `results/seed_<SEED>/<exp>/metrics.json` — per-experiment metrics + training history
- `results/seed_<SEED>/<exp>/history.csv` — long-format (stage, epoch, train, val, lr)
- `results/seed_<SEED>/<exp>/submission.csv` — aligned test-set predictions
- `results/_tables/*_per_seed.csv` — long table, one row per (seed, experiment)
- `results/_tables/*_summary.csv` — cross-seed mean ± std for paper tables
- `data/ofc-2026-ml-challenge/{test_features.csv, test_labels.csv}`

The top section (§§1–5) uses a single `SEED` (default: first seed found, or 42) so you can quickly eyeball one run's curves / spectra.
The bottom section (§7) aggregates across all seeds.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

RESULTS_ROOT = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_ROOT / "_tables"
DATA_DIR = PROJECT_ROOT / "data" / "ofc-2026-ml-challenge"

# Canonical experiment order for figures and tables.
MAIN = ["m1_ours", "m2_mlp", "m3_cnn1d", "m4_transformer"]
TRANSFER = ["m1_ours", "a_t1_no_finetune", "a_t2_no_pretrain", "a_t3_joint"]
ARCH = ["m1_ours", "a_a1_wo_spectral", "a_a2_wo_fourier_kan", "m2_mlp"]
PHYSICS = ["m1_ours", "a_p1_predict_absolute"]


def discover_seeds() -> list[int]:
    seeds = []
    if RESULTS_ROOT.exists():
        for p in RESULTS_ROOT.iterdir():
            if p.is_dir() and p.name.startswith("seed_"):
                try:
                    seeds.append(int(p.name.split("_", 1)[1]))
                except ValueError:
                    pass
    return sorted(seeds)


def seed_dir(seed: int) -> Path:
    return RESULTS_ROOT / f"seed_{seed}"


AVAILABLE_SEEDS = discover_seeds()
# Change SEED here to inspect a different seed in §§1-5.
SEED = AVAILABLE_SEEDS[0] if AVAILABLE_SEEDS else 42

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("PROJECT_ROOT   =", PROJECT_ROOT)
print("available seeds=", AVAILABLE_SEEDS)
print("SEED (for §§1-5) =", SEED)
if AVAILABLE_SEEDS:
    exps = sorted([p.name for p in seed_dir(SEED).glob("*")
                   if p.is_dir() and not p.name.startswith("_")])
    print(f"experiments under seed_{SEED}:", exps)

## 1. Training curves (per stage)

Each subplot is one experiment; within a subplot, separate lines for each training stage (`pretrain`, `finetune`, `joint`). x-axis = global epoch within a stage; y-axis = masked-MSE training / validation loss (log scale).

In [ ]:
def _load_history(exp: str, seed: int = None) -> pd.DataFrame | None:
    s = seed if seed is not None else SEED
    p = seed_dir(s) / exp / "history.csv"
    if not p.exists():
        return None
    df = pd.read_csv(p)
    for col in ["train", "val", "lr"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def plot_training_curves(exps: list[str], title: str = "Training curves"):
    exps_with_data = [e for e in exps if _load_history(e) is not None]
    if not exps_with_data:
        print(f"No history.csv found for any of {exps}")
        return
    n = len(exps_with_data)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5.5 * cols, 3.5 * rows), squeeze=False)
    for i, exp in enumerate(exps_with_data):
        ax = axes[i // cols][i % cols]
        df = _load_history(exp)
        for stage, sub in df.groupby("stage"):
            sub = sub.sort_values("epoch")
            ax.plot(sub["epoch"], sub["train"], label=f"{stage} train", linestyle="-")
            ax.plot(sub["epoch"], sub["val"], label=f"{stage} val", linestyle="--")
        ax.set_title(exp)
        ax.set_xlabel("epoch")
        ax.set_ylabel("masked MSE")
        ax.set_yscale("log")
        ax.legend(fontsize=8)
    # Hide empty axes
    for j in range(len(exps_with_data), rows * cols):
        axes[j // cols][j % cols].axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


plot_training_curves(MAIN, title="Training curves — Main experiments")
plot_training_curves(["m1_ours", "a_a1_wo_spectral", "a_a2_wo_fourier_kan", "a_p1_predict_absolute"],
                     title="Training curves — Architectural / physics ablations")

## 2. Methods comparison — headline bars

MAE and Kaggle Score (both in dB) across the four main methods, broken down by Public / Private split.

In [ ]:
def _load_metrics(exp: str, seed: int = None) -> dict | None:
    s = seed if seed is not None else SEED
    p = seed_dir(s) / exp / "metrics.json"
    if not p.exists():
        return None
    return json.loads(p.read_text())


def compare_methods(exps: list[str], title: str = "Method comparison"):
    rows = []
    for e in exps:
        m = _load_metrics(e)
        if m is None:
            continue
        ov = (m.get("evaluation") or {}).get("overall", {})
        byu = (m.get("evaluation") or {}).get("by_usage", {})
        rows.append({
            "experiment": e,
            "MAE (dB)": ov.get("MAE", np.nan),
            "RMSE (dB)": ov.get("RMSE", np.nan),
            "Kaggle Score (dB)": ov.get("KaggleScore", np.nan),
            "Public MAE (dB)": byu.get("Public", {}).get("MAE", np.nan),
            "Private MAE (dB)": byu.get("Private", {}).get("MAE", np.nan),
        })
    if not rows:
        print(f"No metrics found for {exps} under seed_{SEED}")
        return pd.DataFrame()
    df = pd.DataFrame(rows).set_index("experiment")
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    df[["MAE (dB)"]].plot(kind="bar", ax=axes[0], legend=False, title="Overall MAE (dB)")
    df[["RMSE (dB)"]].plot(kind="bar", ax=axes[1], legend=False, title="Overall RMSE (dB)")
    df[["Kaggle Score (dB)"]].plot(kind="bar", ax=axes[2], legend=False, title="Kaggle Score (dB)")
    for ax in axes:
        ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    return df


df_main = compare_methods(MAIN, title="Main — M-1..M-4")
df_main

In [ ]:
compare_methods(TRANSFER, title="Transfer-learning ablation")
compare_methods(ARCH, title="Architectural ablation")
compare_methods(PHYSICS, title="Physics-baseline ablation (predict offset vs absolute)")
pass

## 3. Category-stratified MAE (dB)

Per-`Category` (aging / shb / unseen / cosmos) and per-`EDFA_type` breakdowns, heat-map style.

In [ ]:
def stratified_heatmap(exps: list[str], key: str, title: str):
    """`key` is 'by_category' or 'by_edfa_type'."""
    rows = {}
    strata: set[str] = set()
    for e in exps:
        m = _load_metrics(e)
        if m is None:
            continue
        grp = (m.get("evaluation") or {}).get(key, {})
        rows[e] = {k: v.get("MAE", np.nan) for k, v in grp.items()}
        strata |= set(rows[e].keys())
    if not rows:
        print(f"No metrics for {exps}")
        return
    ordered_strata = sorted(strata)
    df = pd.DataFrame({e: [rows[e].get(s, np.nan) for s in ordered_strata]
                       for e in rows}, index=ordered_strata).T
    fig, ax = plt.subplots(figsize=(1.5 + 1.2 * len(ordered_strata), 0.6 * len(df) + 1.2))
    im = ax.imshow(df.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(ordered_strata)))
    ax.set_xticklabels(ordered_strata, rotation=30, ha="right")
    ax.set_yticks(range(len(df.index)))
    ax.set_yticklabels(df.index)
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            v = df.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                        color="white" if v > np.nanmedian(df.values) else "black", fontsize=9)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="MAE (dB)")
    fig.tight_layout()
    plt.show()
    return df


stratified_heatmap(MAIN, "by_category", "Main methods — MAE (dB) by Category")
stratified_heatmap(MAIN, "by_edfa_type", "Main methods — MAE (dB) by EDFA type")
stratified_heatmap(TRANSFER, "by_category", "Transfer ablation — MAE (dB) by Category")
pass

## 4. Per-channel error distribution

Box plot of `|y_pred − y_true|` across the 95 WDM channels, masked to activated channels only. Each method contributes one box per channel — here we summarise each channel's error distribution across all test rows.

In [ ]:
N_CHANNELS = 95
_mask_cols = sorted(
    [f"DUT_WSS_activated_channel_index_{i:02d}" for i in range(N_CHANNELS)],
    key=lambda x: int(x.split("_")[-1]),
)
_lbl_cols = [f"calculated_gain_spectra_{i:02d}" for i in range(N_CHANNELS)]


def _test_frames():
    test_features = pd.read_csv(DATA_DIR / "test_features.csv")
    test_labels = pd.read_csv(DATA_DIR / "test_labels.csv")
    return test_features, test_labels


def _per_channel_abs_err(exp: str, test_features: pd.DataFrame, test_labels: pd.DataFrame,
                         seed: int = None) -> np.ndarray | None:
    s = seed if seed is not None else SEED
    p = seed_dir(s) / exp / "submission.csv"
    if not p.exists():
        return None
    sub = pd.read_csv(p)
    # Align on ID
    merged_pred = test_features[["ID"]].merge(sub[["ID"] + _lbl_cols], on="ID", how="left")
    y_pred = merged_pred[_lbl_cols].values.astype(np.float32)
    merged_true = test_features[["ID"]].merge(test_labels[["ID"] + _lbl_cols], on="ID", how="left")
    y_true = merged_true[_lbl_cols].values.astype(np.float32)
    mask = test_features[_mask_cols].values.astype(np.float32)
    err = np.abs(y_pred - y_true)
    err[mask <= 0] = np.nan  # drop inactive channels
    return err


def plot_channel_error_medians(exps: list[str]):
    test_features, test_labels = _test_frames()
    fig, ax = plt.subplots(figsize=(12, 4))
    for e in exps:
        err = _per_channel_abs_err(e, test_features, test_labels)
        if err is None:
            continue
        med = np.nanmedian(err, axis=0)
        p95 = np.nanquantile(err, 0.95, axis=0)
        ax.plot(range(N_CHANNELS), med, label=f"{e} (median)", linewidth=1.2)
        ax.fill_between(range(N_CHANNELS), med, p95, alpha=0.12)
    ax.set_xlabel("WDM channel index (0..94)")
    ax.set_ylabel("|err| (dB)")
    ax.set_title("Per-channel absolute error: median (line) and 95th percentile (shaded)")
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    plt.show()


plot_channel_error_medians(MAIN)

## 5. Typical prediction spectra

Pick one sample per `Category` and overlay ground-truth vs predicted gain for the 4 main methods.

In [ ]:
def plot_example_spectra(exps: list[str], seed: int = 0):
    test_features, test_labels = _test_frames()
    rng = np.random.default_rng(seed)

    categories = [c for c in ["aging", "shb", "unseen", "cosmos"] if c in test_features["Category"].unique()]
    fig, axes = plt.subplots(len(categories), 1, figsize=(10, 2.6 * len(categories)), squeeze=False)

    # Align everything on the test_features order once.
    merged_true = test_features[["ID", "Category"]].merge(test_labels[["ID"] + _lbl_cols], on="ID", how="left")
    y_true_all = merged_true[_lbl_cols].values
    mask_all = test_features[_mask_cols].values

    preds_by_exp = {}
    for e in exps:
        err = _per_channel_abs_err(e, test_features, test_labels)  # just to hit the file + align
        p = seed_dir(SEED) / e / "submission.csv"
        if not p.exists():
            continue
        sub = pd.read_csv(p)
        merged = test_features[["ID"]].merge(sub[["ID"] + _lbl_cols], on="ID", how="left")
        preds_by_exp[e] = merged[_lbl_cols].values

    for i, cat in enumerate(categories):
        idxs = merged_true.index[merged_true["Category"] == cat].tolist()
        if not idxs:
            continue
        pick = int(rng.choice(idxs))
        m = mask_all[pick].astype(bool)
        ax = axes[i][0]
        ax.plot(np.arange(N_CHANNELS)[m], y_true_all[pick][m], "k-", linewidth=2, label="ground truth")
        for e, preds in preds_by_exp.items():
            ax.plot(np.arange(N_CHANNELS)[m], preds[pick][m], linestyle="--", alpha=0.8, label=e)
        ax.set_title(f"Category = {cat}   (row #{pick}, {m.sum()} channels active)")
        ax.set_xlabel("WDM channel")
        ax.set_ylabel("gain (dB)")
        ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    plt.show()


plot_example_spectra(MAIN)

## 6. Aggregate tables at a glance

Pretty-print the auto-generated paper tables under `results/_tables/`.

In [ ]:
from IPython.display import display

# Prefer the new multi-seed summary tables (`*_summary.csv`).  Fall back to the
# per-seed long tables if summaries are missing.
_candidates = [
    ("table1_main_summary.csv",     "table1_main_per_seed.csv"),
    ("table2_transfer_summary.csv", "table2_transfer_per_seed.csv"),
    ("table3_arch_summary.csv",     "table3_arch_per_seed.csv"),
    ("table4_physics_summary.csv",  "table4_physics_per_seed.csv"),
    ("figure3_data_scale_summary.csv", "figure3_data_scale_per_seed.csv"),
]
for primary, fallback in _candidates:
    p = TABLES_DIR / primary
    if not p.exists():
        p = TABLES_DIR / fallback
        if not p.exists():
            continue
    df = pd.read_csv(p)
    keep = [c for c in df.columns
            if c in ("experiment", "seed", "n_seeds", "seeds", "model", "stages",
                     "params", "cosmos_ratio", "kaggle_ratio", "predict_absolute",
                     "ratio", "axis")
            or c.endswith("_str")
            or (("_dB" in c) and not c.endswith(("_mean", "_std")))]
    print(f"=== {p.name} ===")
    display(df[keep])

## 7. Cross-seed mean ± std (all seeds)

Uses `results/_tables/*_summary.csv` (produced by `python scripts/aggregate_results.py`).
Bars show the mean, error bars show std across seeds.

In [ ]:
def plot_cross_seed(summary_csv: str, title: str):
    p = TABLES_DIR / summary_csv
    if not p.exists():
        print(f"[skip] {summary_csv} not found; run scripts/aggregate_results.py first.")
        return None
    df = pd.read_csv(p)
    if df.empty:
        print(f"[skip] {summary_csv} has 0 rows.")
        return None
    metrics = [("MAE_dB", "Overall MAE (dB)"),
               ("RMSE_dB", "Overall RMSE (dB)"),
               ("KaggleScore_dB", "Kaggle Score (dB)")]
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 3.8))
    x = np.arange(len(df))
    for ax, (m, label) in zip(axes, metrics):
        mean_col, std_col = f"{m}_mean", f"{m}_std"
        if mean_col not in df or std_col not in df:
            continue
        ax.bar(x, df[mean_col], yerr=df[std_col], capsize=4)
        ax.set_xticks(x)
        ax.set_xticklabels(df["experiment"], rotation=25, ha="right")
        ax.set_ylabel(label)
        ax.set_title(label)
    n_seeds_str = df["n_seeds"].astype(int).tolist() if "n_seeds" in df else []
    fig.suptitle(f"{title}  (n_seeds per cell: {n_seeds_str})")
    fig.tight_layout()
    plt.show()
    return df


plot_cross_seed("table1_main_summary.csv",     "Main results")
plot_cross_seed("table2_transfer_summary.csv", "Transfer ablation")
plot_cross_seed("table3_arch_summary.csv",     "Architectural ablation")
plot_cross_seed("table4_physics_summary.csv",  "Physics-baseline ablation")

## 8. Training curves overlaid across seeds (stability check)

For a chosen experiment, overlay the per-epoch train / val loss from every seed available. Flat convergence across seeds = stable optimisation; wild divergence = unstable.

In [ ]:
def plot_curves_across_seeds(exp: str, stages: tuple[str, ...] = ("pretrain", "finetune")):
    if not AVAILABLE_SEEDS:
        print("No seeds discovered."); return
    fig, axes = plt.subplots(1, len(stages), figsize=(6 * len(stages), 3.6), squeeze=False)
    any_plotted = False
    for ax, stage in zip(axes[0], stages):
        for seed in AVAILABLE_SEEDS:
            h = _load_history(exp, seed=seed)
            if h is None:
                continue
            sub = h[h["stage"] == stage].sort_values("epoch")
            if sub.empty:
                continue
            ax.plot(sub["epoch"], sub["train"], label=f"seed={seed} train", alpha=0.8)
            ax.plot(sub["epoch"], sub["val"],   label=f"seed={seed} val",   linestyle="--", alpha=0.8)
            any_plotted = True
        ax.set_title(f"{exp}  / stage={stage}")
        ax.set_xlabel("epoch")
        ax.set_ylabel("masked MSE")
        ax.set_yscale("log")
        ax.legend(fontsize=7, ncol=2)
    if not any_plotted:
        print(f"[skip] no history found for {exp}")
    fig.tight_layout()
    plt.show()


plot_curves_across_seeds("m1_ours")
plot_curves_across_seeds("a_a1_wo_spectral")